In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/varunika18/complaints-data/complaints.csv


In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

print("Libraries imported successfully.")

Libraries imported successfully.


In [3]:
df = pd.read_csv("/kaggle/input/datasets/varunika18/complaints-data/complaints.csv")

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

Dataset shape: (18338, 4)
Columns: ['category', 'sub_category', 'crimeaditionalinfo', 'is_short']


In [4]:
print(df.head())
print("\nMissing values:")
print(df.isnull().sum())


                                category  \
0  Online and Social Media Related Crime   
1         Cyber Attack/ Dependent Crimes   
2  Online and Social Media Related Crime   
3                 Online Financial Fraud   
4  Online and Social Media Related Crime   

                           sub_category  \
0  Provocative Speech for unlawful acts   
1                    Hacking/Defacement   
2                      Online Job Fraud   
3  DebitCredit Card FraudSim Swap Fraud   
4             FakeImpersonating Profile   

                                  crimeaditionalinfo  is_short  
0  i was sent a msg on whatsapp and she said she ...     False  
1  i never imagined something like this could hap...     False  
2  online job add was posted in facebook in group...     False  
3  citizen details namenaresh pshalirwaas distchu...     False  
4  one fake id has been created in my facebook ac...     False  

Missing values:
category              0
sub_category          0
crimeaditionalinfo   

In [5]:
print(df["category"].value_counts())
print("\nNumber of categories:", df["category"].nunique())

category
Online and Social Media Related Crime                   4746
Online Financial Fraud                                  4255
Cyber Attack/ Dependent Crimes                          3608
Hacking  Damage to computercomputer system etc          1587
Sexually Obscene material                               1543
Any Other Cyber Crime                                   1543
Cryptocurrency Crime                                     465
Child Pornography CPChild Sexual Abuse Material CSAM     350
RapeGang Rape RGRSexually Abusive Content                241
Name: count, dtype: int64

Number of categories: 9


In [6]:
df["category"] = df["category"].replace({
    "Hacking  Damage to computercomputer system etc":
        "Hacking / Computer System Damage",

    "Online and Social Media Related Crime":
        "Online & Social Media Crime",

    "RapeGang Rape RGRSexually Abusive Content":
        "Rape / Sexually Abusive Content",

    "Child Pornography CPChild Sexual Abuse Material CSAM":
        "Child Sexual Abuse Material"
})

print(df["category"].value_counts())

category
Online & Social Media Crime         4746
Online Financial Fraud              4255
Cyber Attack/ Dependent Crimes      3608
Hacking / Computer System Damage    1587
Sexually Obscene material           1543
Any Other Cyber Crime               1543
Cryptocurrency Crime                 465
Child Sexual Abuse Material          350
Rape / Sexually Abusive Content      241
Name: count, dtype: int64


In [7]:
X = df["crimeaditionalinfo"]
y = df["category"]

print("Input shape:", X.shape)
print("Target shape:", y.shape)

print("\nSample complaint:")
print(X.iloc[0])

print("\nSample target:")
print(y.iloc[0])

Input shape: (18338,)
Target shape: (18338,)

Sample complaint:
i was sent a msg on whatsapp and she said she is a girl and want to be my girlfriend and she did a video call and asked me to be nude then she or he did a screen recording and blackmailed me to upload it and asked for money she also showed me some screenshots of uploading the videos

Sample target:
Online & Social Media Crime


In [8]:
duplicate_count = X.duplicated().sum()

print("Duplicate complaint texts:", duplicate_count)

Duplicate complaint texts: 0


In [9]:
from sklearn.model_selection import train_test_split

# First: 80% train+validation, 20% test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# Second: split the 80% into 60% train and 20% validation
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval,
    y_trainval,
    test_size=0.25,
    random_state=42,
    stratify=y_trainval
)

print("Training:", len(X_train))
print("Validation:", len(X_val))
print("Testing:", len(X_test))

Training: 11002
Validation: 3668
Testing: 3668


In [10]:
print("Training class distribution:")
print(y_train.value_counts())

Training class distribution:
category
Online & Social Media Crime         2848
Online Financial Fraud              2553
Cyber Attack/ Dependent Crimes      2164
Hacking / Computer System Damage     952
Sexually Obscene material            926
Any Other Cyber Crime                925
Cryptocurrency Crime                 279
Child Sexual Abuse Material          210
Rape / Sexually Abusive Content      145
Name: count, dtype: int64


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_lr = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=50000,
    min_df=2,
    sublinear_tf=True
)

X_train_lr = tfidf_lr.fit_transform(X_train)
X_val_lr = tfidf_lr.transform(X_val)
X_test_lr = tfidf_lr.transform(X_test)

print("Training TF-IDF shape:", X_train_lr.shape)
print("Validation TF-IDF shape:", X_val_lr.shape)
print("Testing TF-IDF shape:", X_test_lr.shape)

Training TF-IDF shape: (11002, 42262)
Validation TF-IDF shape: (3668, 42262)
Testing TF-IDF shape: (3668, 42262)


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import make_union

vec = make_union(
    TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True, max_features=100000),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=3, sublinear_tf=True, max_features=150000)
)

X_train_f = vec.fit_transform(X_train)
X_val_f   = vec.transform(X_val)
X_test_f  = vec.transform(X_test)

In [13]:
import numpy as np
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, f1_score

SEX = ["Sexually Obscene material", "Child Sexual Abuse Material", "Rape / Sexually Abusive Content"]

def scheme_A(c):   # merge the 3 sexual-content classes
    return "Sexual Content" if c in SEX else c

def scheme_B(c):   # A + drop the catch-all class
    return None if c == "Any Other Cyber Crime" else scheme_A(c)

def scheme_C(c):   # B + merge sexual content with social media (sextortion overlap)
    c = scheme_B(c)
    return "Online & Social Media Crime" if c == "Sexual Content" else c

def scheme_D(c):   # C + merge crypto into financial fraud
    c = scheme_C(c)
    return "Online Financial Fraud" if c == "Cryptocurrency Crime" else c

for name, f in [("A", scheme_A), ("B", scheme_B), ("C", scheme_C), ("D", scheme_D)]:
    ytr = y_train.map(f); yva = y_val.map(f)
    mtr = ytr.notna().values; mva = yva.notna().values
    m = LinearSVC(C=0.5, class_weight="balanced").fit(X_train_f[mtr], ytr[mtr])
    p = m.predict(X_val_f[mva])
    print(f"Scheme {name}: classes={ytr[mtr].nunique()}  acc={accuracy_score(yva[mva], p):.4f}  "
          f"wF1={f1_score(yva[mva], p, average='weighted'):.4f}  mF1={f1_score(yva[mva], p, average='macro'):.4f}")

Scheme A: classes=7  acc=0.6805  wF1=0.6750  mF1=0.6419
Scheme B: classes=6  acc=0.7508  wF1=0.7494  mF1=0.7372
Scheme C: classes=5  acc=0.8431  wF1=0.8429  mF1=0.8152
Scheme D: classes=4  acc=0.8476  wF1=0.8472  mF1=0.8220


In [14]:
import re, numpy as np, pandas as pd
from scipy.sparse import vstack
from sklearn.svm import LinearSVC
from sklearn.pipeline import make_union
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, classification_report

# 1. clean text
def clean(t):
    t = str(t).lower()
    t = re.sub(r"\d{10,}", " longnumber ", t)
    t = re.sub(r"\d+", " num ", t)
    return re.sub(r"\s+", " ", re.sub(r"[^a-z\s]", " ", t)).strip()

Xtr, Xva, Xte = X_train.apply(clean), X_val.apply(clean), X_test.apply(clean)

# 2. ONE vectorizer, fitted once on train only, then used for all three sets
vec = make_union(
    TfidfVectorizer(ngram_range=(1,2), min_df=2, sublinear_tf=True),
    TfidfVectorizer(analyzer="char_wb", ngram_range=(3,5), min_df=3, sublinear_tf=True, max_features=150000))
Ftr = vec.fit_transform(Xtr)
Fva = vec.transform(Xva)
Fte = vec.transform(Xte)
print("Features:", Ftr.shape[1], Fva.shape[1], Fte.shape[1])   # all three must match

# 3. label scheme C
SEX = ["Sexually Obscene material", "Child Sexual Abuse Material", "Rape / Sexually Abusive Content"]
def scheme_C(c):
    if c == "Any Other Cyber Crime": return None
    if c in SEX: return "Online & Social Media Crime"
    return c

ytr_all = pd.concat([y_train, y_val]).map(scheme_C)
mask = ytr_all.notna().values
yte = y_test.map(scheme_C)
mte = yte.notna().values

# 4. train on train+val, test once
final = LinearSVC(C=0.5, class_weight="balanced")
final.fit(vstack([Ftr, Fva])[mask], ytr_all[mask])
pt = final.predict(Fte[mte])

print("Test accuracy   :", round(accuracy_score(yte[mte], pt), 4))
print("Test weighted F1:", round(f1_score(yte[mte], pt, average="weighted"), 4))
print("Test macro F1   :", round(f1_score(yte[mte], pt, average="macro"), 4))
print(classification_report(yte[mte], pt, zero_division=0))

Features: 106303 106303 106303
Test accuracy   : 0.8598
Test weighted F1: 0.86
Test macro F1   : 0.8442
                                  precision    recall  f1-score   support

            Cryptocurrency Crime       0.87      0.83      0.85        93
  Cyber Attack/ Dependent Crimes       1.00      1.00      1.00       722
Hacking / Computer System Damage       0.69      0.73      0.71       317
     Online & Social Media Crime       0.85      0.85      0.85      1376
          Online Financial Fraud       0.83      0.81      0.82       851

                        accuracy                           0.86      3359
                       macro avg       0.85      0.84      0.84      3359
                    weighted avg       0.86      0.86      0.86      3359



In [15]:
import pandas as pd
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_true = yte[mte]
y_pred = pt

results = pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 Score"],
    "Score": [
        accuracy_score(y_true, y_pred),
        precision_score(y_true, y_pred, average="weighted", zero_division=0),
        recall_score(y_true, y_pred, average="weighted", zero_division=0),
        f1_score(y_true, y_pred, average="weighted", zero_division=0),
    ],
}).round(4)

display(results)
results.to_excel("model_metrics.xlsx", index=False)
print("Saved: model_metrics.xlsx")

,Metric,Score
0,Accuracy,0.8598
1,Precision,0.8605
2,Recall,0.8598
3,F1 Score,0.8600


Saved: model_metrics.xlsx


In [16]:
import joblib

joblib.dump(final, "/kaggle/working/crime_type_model.pkl")
joblib.dump(vec, "/kaggle/working/crime_type_tfidf.pkl")

print("Crime Type model saved successfully!")
print("crime_type_model.pkl")
print("crime_type_tfidf.pkl")

Crime Type model saved successfully!
crime_type_model.pkl
crime_type_tfidf.pkl
